In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# ============================================================
# Multi-seed 设置
# ============================================================
seeds = [42, 123, 777, 1004, 2024]
# [11, 22, 33, 44, 55]
patience = 20

all_seed_best_1mae = []
all_seed_best_epoch = []


# ============================================================
# 你最终只保留这一个文件
# ============================================================
best_model_path = os.path.join(
    "./save_swintransformer_folder",
    "best_model.pth"
)


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:1' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-21 01:33:30,531]::2043440756::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:1


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder'
print(1)

1


In [5]:
validation_set_data=getdata("/data1/qx/mean_based_arg/KETI/data/scene3_valid_set30s.dat","valid")
print(1)
train_set_data=getdata("/data1/qx/mean_based_arg/KETI/data/scene3_train_set30s.dat","train")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)/120
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[3],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[3],transform=transform)
train_dataloader = DataLoader(dataset=train_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
val_dataloader = DataLoader(dataset=val_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
max_value=0
print(1)

1


In [10]:
import os
import random
import numpy as np
import torch
from tqdm import tqdm


# ============================================================
# 固定随机种子
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 保证实验尽可能可复现
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# OCEAN 分离
# ============================================================
def extract_ocean(data, batch_size):
    o = []
    c = []
    e = []
    a = []
    n = []

    if batch_size > 1:

        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())

    elif batch_size == 1:

        data = data.squeeze()

        o.append(data[0].item())
        c.append(data[1].item())
        e.append(data[2].item())
        a.append(data[3].item())
        n.append(data[4].item())

    o = torch.tensor(o)
    c = torch.tensor(c)
    e = torch.tensor(e)
    a = torch.tensor(a)
    n = torch.tensor(n)

    return o, c, e, a, n




# ============================================================
# Multi-seed training
# ============================================================
for seed in seeds:

    print("\n" + "=" * 80)
    print(f"Start Seed: {seed}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. 设置当前 seed
    # --------------------------------------------------------
    set_seed(seed)


    # --------------------------------------------------------
    # 2. 每个 seed 都重新初始化模型
    # --------------------------------------------------------
    model = SwinTransformer3D()
    model.to(device)

    criterion = torch.nn.L1Loss().to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr
    )


    # --------------------------------------------------------
    # 如果是正常 multi-seed 实验，
    # 建议 test_flage=False
    # 否则会加载之前模型，破坏 seed 独立性
    # --------------------------------------------------------
    if test_flage == True:

        checkpoint = torch.load(
            best_model_path,
            map_location=device
        )

        model.load_state_dict(
            checkpoint["model"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer"]
        )


    # --------------------------------------------------------
    # 每个 seed 都必须重新初始化
    # --------------------------------------------------------
    max_value = float('inf')

    no_improve_count = 0

    best_epoch = 0


    with torch.cuda.device(1):

        for i in range(start_epoch, epochs):

            # =================================================
            # Train
            # =================================================
            model.train()

            train_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0


            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} Train Epoch {i + 1}"
            ):

                fullshot = fullshot.permute(
                    0, 4, 1, 2, 3
                )

                big_five_data = big_five_data.permute(
                    0, 2, 1
                )

                big_five_data = big_five_data.squeeze()


                audiodata = audiodata.to(device)

                fullshot = fullshot.to(device)

                big_five_data = big_five_data.to(device)


                optimizer.zero_grad()


                hypothesis = model(fullshot, audiodata)


                loss = criterion(
                    hypothesis,
                    big_five_data
                )


                loss.backward()

                optimizer.step()


                train_avg_loss += loss.item()


            train_avg_loss = (
                train_avg_loss /
                len(train_dataloader)
            )


            # =================================================
            # Validation
            # =================================================
            # model.eval()


            val_avg_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0


            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} Val Epoch {i + 1}"
                ):

                    fullshot = fullshot.permute(
                        0, 4, 1, 2, 3
                    )


                    big_five_data = big_five_data.permute(
                        0, 2, 1
                    )

                    big_five_data = big_five_data.squeeze()


                    audiodata = audiodata.to(device)

                    fullshot = fullshot.to(device)

                    big_five_data = big_five_data.to(device)


                    hypothesis = model(fullshot, audiodata)


                    val_loss = criterion(
                        hypothesis,
                        big_five_data
                    )


                    hypothesiso, \
                    hypothesisc, \
                    hypothesise, \
                    hypothesisa, \
                    hypothesisn = extract_ocean(
                        hypothesis,
                        len(fullshot)
                    )


                    big_five_datao, \
                    big_five_datac, \
                    big_five_datae, \
                    big_five_dataa, \
                    big_five_datan = extract_ocean(
                        big_five_data,
                        len(fullshot)
                    )


                    val_o_loss = criterion(
                        hypothesiso,
                        big_five_datao
                    )

                    val_c_loss = criterion(
                        hypothesisc,
                        big_five_datac
                    )

                    val_e_loss = criterion(
                        hypothesise,
                        big_five_datae
                    )

                    val_a_loss = criterion(
                        hypothesisa,
                        big_five_dataa
                    )

                    val_n_loss = criterion(
                        hypothesisn,
                        big_five_datan
                    )


                    val_avg_loss += val_loss.item()

                    val_avg_o_loss += val_o_loss.item()

                    val_avg_c_loss += val_c_loss.item()

                    val_avg_e_loss += val_e_loss.item()

                    val_avg_a_loss += val_a_loss.item()

                    val_avg_n_loss += val_n_loss.item()


            val_avg_loss = (
                val_avg_loss /
                len(val_dataloader)
            )

            val_avg_o_loss = (
                val_avg_o_loss /
                len(val_dataloader)
            )

            val_avg_c_loss = (
                val_avg_c_loss /
                len(val_dataloader)
            )

            val_avg_e_loss = (
                val_avg_e_loss /
                len(val_dataloader)
            )

            val_avg_a_loss = (
                val_avg_a_loss /
                len(val_dataloader)
            )

            val_avg_n_loss = (
                val_avg_n_loss /
                len(val_dataloader)
            )


            # =================================================
            # Best model
            #
            # 注意：
            # 永远保存到同一个 best_model.pth
            # 所以前一次会被覆盖
            # =================================================
            if val_avg_loss < max_value:

                max_value = val_avg_loss

                no_improve_count = 0

                best_epoch = i + 1


                torch.save(
                    {
                        'seed': seed,

                        'epoch': i + 1,

                        'model': model.state_dict(),

                        'optimizer': optimizer.state_dict(),

                        'loss': val_avg_loss,

                        'best_1_mae': 1 - val_avg_loss
                    },

                    best_model_path
                )


                logger.info(
                    '[Seed {}] New Best Model Saved | '
                    'Epoch: {} | '
                    '1-MAE: {:.4f}'.format(
                        seed,
                        i + 1,
                        1 - val_avg_loss
                    )
                )


            else:

                no_improve_count += 1


            # =================================================
            # 当前指标
            # =================================================
            current_1_mae = (
                1 - val_avg_loss
            )


            best_1_mae = (
                1 - max_value
            )


            logger.debug(
                'Seed: {} , '
                'Epoch: {} , '
                '1-MAE: {:.4f} , '
                'best 1-MAE: {:.4f} , '
                'patience: {}/{} , '
                '1-Training Loss: {:.4f} , '
                '1-val_avg_o_loss: {:.4f} , '
                '1-val_avg_c_loss: {:.4f} , '
                '1-val_avg_e_loss: {:.4f} , '
                '1-val_avg_a_loss: {:.4f} , '
                '1-val_avg_n_loss: {:.4f}'.format(

                    seed,

                    i + 1,

                    current_1_mae,

                    best_1_mae,

                    no_improve_count,

                    patience,

                    1 - train_avg_loss,

                    1 - val_avg_o_loss,

                    1 - val_avg_c_loss,

                    1 - val_avg_e_loss,

                    1 - val_avg_a_loss,

                    1 - val_avg_n_loss
                )
            )


            # =================================================
            # Early stopping
            # =================================================
            if no_improve_count >= patience:

                logger.info(
                    '[Seed {}] Early stopping triggered '
                    'at epoch {}. '
                    'Best epoch: {}. '
                    'Best 1-MAE: {:.4f}, '
                    'patience={}.'.format(

                        seed,

                        i + 1,

                        best_epoch,

                        best_1_mae,

                        patience
                    )
                )

                break


            torch.cuda.empty_cache()


    # ========================================================
    # 当前 seed 训练结束
    # ========================================================
    all_seed_best_1mae.append(
        best_1_mae
    )

    all_seed_best_epoch.append(
        best_epoch
    )


    print(
        f"\nSeed {seed} finished | "
        f"Best Epoch = {best_epoch} | "
        f"Best 1-MAE = {best_1_mae:.4f}"
    )


    # ========================================================
    # 删除模型，准备下一个 seed
    # ========================================================
    del model
    del optimizer

    torch.cuda.empty_cache()


# ============================================================
# 所有 seed 训练完成
# ============================================================
all_seed_best_1mae = np.array(
    all_seed_best_1mae
)


mean_1mae = np.mean(
    all_seed_best_1mae
)


std_1mae = np.std(
    all_seed_best_1mae,
    ddof=1
)


print("\n" + "=" * 80)
print("FINAL MULTI-SEED RESULTS")
print("=" * 80)


for seed, epoch, score in zip(
    seeds,
    all_seed_best_epoch,
    all_seed_best_1mae
):

    print(
        f"Seed {seed:>3} | "
        f"Best Epoch: {epoch:>3} | "
        f"Best 1-MAE: {score:.4f}"
    )


print("-" * 80)

print(
    f"Mean 1-MAE: {mean_1mae:.4f}"
)

print(
    f"Std  1-MAE: {std_1mae:.4f}"
)

print(
    f"Final Result: "
    f"{mean_1mae:.4f} ± {std_1mae:.4f}"
)

print("=" * 80)

print(
    f"\nFinal saved model: "
    f"{best_model_path}"
)


Start Seed: 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


Seed 42 Val Epoch 1: 100%|██████████| 337/337 [00:33<00:00, 10.10it/s]
[2026-09-21 01:38:58,446]::3110008408::INFO::[Seed 42] New Best Model Saved | Epoch: 1 | 1-MAE: 0.8789
[2026-09-21 01:38:58,447]::3110008408::DEBUG::Seed: 42 , Epoch: 1 , 1-MAE: 0.8789 , best 1-MAE: 0.8789 , patience: 0/20 , 1-Training Loss: 0.8727 , 1-val_avg_o_loss: 0.8809 , 1-val_avg_c_loss: 0.8813 , 1-val_avg_e_loss: 0.8846 , 1-val_avg_a_loss: 0.9209 , 1-val_avg_n_loss: 0.8266
Seed 42 Val Epoch 2: 100%|██████████| 337/337 [00:33<00:00, 10.15it/s]
[2026-09-21 01:44:19,660]::3110008408::INFO::[Seed 42] New Best Model Saved | Epoch: 2 | 1-MAE: 0.8908
[2026-09-21 01:44:19,661]::3110008408::DEBUG::Seed: 42 , Epoch: 2 , 1-MAE: 0.8908 , best 1-MAE: 0.8908 , patience: 0/20 , 1-Training Loss: 0.8951 , 1-val_avg_o_loss: 0.8257 , 1-val_avg_c_loss: 0.9076 , 1-val_avg_e_loss: 0.8982 , 1-val_avg_a_loss: 0.9141 , 1-val_avg_n_loss: 0.9082
Seed 42 Val Epoch 3: 100%|██████████| 337/337 [00:33<00:00, 10.15it/s]
[2026-09-21 01:49:4


Seed 42 finished | Best Epoch = 8 | Best 1-MAE = 0.9125

Start Seed: 123
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


Seed 123 Val Epoch 1: 100%|██████████| 337/337 [00:33<00:00, 10.09it/s]
[2026-09-21 04:07:58,784]::3110008408::INFO::[Seed 123] New Best Model Saved | Epoch: 1 | 1-MAE: 0.8691
[2026-09-21 04:07:58,786]::3110008408::DEBUG::Seed: 123 , Epoch: 1 , 1-MAE: 0.8691 , best 1-MAE: 0.8691 , patience: 0/20 , 1-Training Loss: 0.8750 , 1-val_avg_o_loss: 0.9145 , 1-val_avg_c_loss: 0.7440 , 1-val_avg_e_loss: 0.8986 , 1-val_avg_a_loss: 0.8758 , 1-val_avg_n_loss: 0.9124
Seed 123 Val Epoch 2: 100%|██████████| 337/337 [00:33<00:00, 10.02it/s]
[2026-09-21 04:13:22,910]::3110008408::INFO::[Seed 123] New Best Model Saved | Epoch: 2 | 1-MAE: 0.9031
[2026-09-21 04:13:22,911]::3110008408::DEBUG::Seed: 123 , Epoch: 2 , 1-MAE: 0.9031 , best 1-MAE: 0.9031 , patience: 0/20 , 1-Training Loss: 0.8932 , 1-val_avg_o_loss: 0.9134 , 1-val_avg_c_loss: 0.9120 , 1-val_avg_e_loss: 0.8527 , 1-val_avg_a_loss: 0.9292 , 1-val_avg_n_loss: 0.9082
Seed 123 Val Epoch 3: 100%|██████████| 337/337 [00:33<00:00, 10.02it/s]
[2026-09-21 


Seed 123 finished | Best Epoch = 12 | Best 1-MAE = 0.9072

Start Seed: 777
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


...


Seed 777 finished | Best Epoch = 9 | Best 1-MAE = 0.9131

Start Seed: 1004
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


Seed 1004 Val Epoch 1: 100%|██████████| 337/337 [00:34<00:00,  9.81it/s]
[2026-09-21 09:34:49,395]::3110008408::INFO::[Seed 1004] New Best Model Saved | Epoch: 1 | 1-MAE: 0.9054
[2026-09-21 09:34:49,396]::3110008408::DEBUG::Seed: 1004 , Epoch: 1 , 1-MAE: 0.9054 , best 1-MAE: 0.9054 , patience: 0/20 , 1-Training Loss: 0.8739 , 1-val_avg_o_loss: 0.9152 , 1-val_avg_c_loss: 0.9081 , 1-val_avg_e_loss: 0.8997 , 1-val_avg_a_loss: 0.9180 , 1-val_avg_n_loss: 0.8862
Seed 1004 Val Epoch 2: 100%|██████████| 337/337 [00:33<00:00, 10.02it/s]
[2026-09-21 09:40:10,155]::3110008408::DEBUG::Seed: 1004 , Epoch: 2 , 1-MAE: 0.9054 , best 1-MAE: 0.9054 , patience: 1/20 , 1-Training Loss: 0.8952 , 1-val_avg_o_loss: 0.9114 , 1-val_avg_c_loss: 0.9087 , 1-val_avg_e_loss: 0.8876 , 1-val_avg_a_loss: 0.9144 , 1-val_avg_n_loss: 0.9047
Seed 1004 Val Epoch 3: 100%|██████████| 337/337 [00:33<00:00, 10.06it/s]
[2026-09-21 09:45:31,038]::3110008408::DEBUG::Seed: 1004 , Epoch: 3 , 1-MAE: 0.9039 , best 1-MAE: 0.9054 , pat


Seed 1004 finished | Best Epoch = 6 | Best 1-MAE = 0.9067

Start Seed: 2024
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


Seed 2024 Val Epoch 1: 100%|██████████| 337/337 [00:34<00:00,  9.85it/s]
[2026-09-21 11:54:04,817]::3110008408::INFO::[Seed 2024] New Best Model Saved | Epoch: 1 | 1-MAE: 0.8869
[2026-09-21 11:54:04,819]::3110008408::DEBUG::Seed: 2024 , Epoch: 1 , 1-MAE: 0.8869 , best 1-MAE: 0.8869 , patience: 0/20 , 1-Training Loss: 0.8721 , 1-val_avg_o_loss: 0.9030 , 1-val_avg_c_loss: 0.8725 , 1-val_avg_e_loss: 0.8216 , 1-val_avg_a_loss: 0.9300 , 1-val_avg_n_loss: 0.9077
Seed 2024 Val Epoch 2: 100%|██████████| 337/337 [00:33<00:00, 10.11it/s]
[2026-09-21 11:59:28,527]::3110008408::INFO::[Seed 2024] New Best Model Saved | Epoch: 2 | 1-MAE: 0.8874
[2026-09-21 11:59:28,529]::3110008408::DEBUG::Seed: 2024 , Epoch: 2 , 1-MAE: 0.8874 , best 1-MAE: 0.8874 , patience: 0/20 , 1-Training Loss: 0.8942 , 1-val_avg_o_loss: 0.9115 , 1-val_avg_c_loss: 0.8593 , 1-val_avg_e_loss: 0.8950 , 1-val_avg_a_loss: 0.8803 , 1-val_avg_n_loss: 0.8906
Seed 2024 Val Epoch 3: 100%|██████████| 337/337 [00:33<00:00, 10.11it/s]
[2026


Seed 2024 finished | Best Epoch = 10 | Best 1-MAE = 0.9061

FINAL MULTI-SEED RESULTS
Seed  42 | Best Epoch:   8 | Best 1-MAE: 0.9125
Seed 123 | Best Epoch:  12 | Best 1-MAE: 0.9072
Seed 777 | Best Epoch:   9 | Best 1-MAE: 0.9131
Seed 1004 | Best Epoch:   6 | Best 1-MAE: 0.9067
Seed 2024 | Best Epoch:  10 | Best 1-MAE: 0.9061
--------------------------------------------------------------------------------
Mean 1-MAE: 0.9091
Std  1-MAE: 0.0034
Final Result: 0.9091 ± 0.0034

Final saved model: ./save_swintransformer_folder/best_model.pth
